In [1]:
from flask import Flask, request, jsonify
import requests
import urllib.parse

In [ ]:
app = Flask(__name__)

def build_wikipedia_api_url(query):
    """
    Converts a user query like 'machine learning' into a proper Wikipedia API URL.
    """
    base_api_url = "https://en.wikipedia.org/api/rest_v1/page/summary/"
    formatted_query = urllib.parse.quote(query.strip().replace(" ", "_"))
    return base_api_url + formatted_query

@app.route('/scrape', methods=['POST'])
def scrape():
    data = request.json
    query = data.get('query')

    if not query:
        return jsonify({"error": "Query is required"}), 400

    wiki_api_url = build_wikipedia_api_url(query)

    # Make a request to Wikipedia REST API
    response = requests.get(wiki_api_url)

    if response.status_code != 200:
        return jsonify({"error": "Wikipedia API request failed", "status_code": response.status_code}), 500

    data = response.json()
    content = data.get("extract", "")
    source_url = data.get("content_urls", {}).get("desktop", {}).get("page", "")

    return jsonify({
        "content": content,
        "source_url": source_url
    })

if __name__ == '__main__':
    app.run(debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [04/May/2025 00:19:50] "POST /scrape HTTP/1.1" 200 -
